# UNet AudioSet spectrogram dashboard

Run this notebook top to bottom.

1. **Setup** — paths and imports
2. **Preprocess** — UNet on every clip; writes stems + spectrogram PNGs under `datasets/audioset/processed/` (skips unchanged clips)
3. **Dashboard** — pick a sub-label; grid shows **voice** (top) and **background** (bottom) specs per clip. Overlap between them often means confusing separation. Select a clip to hear mix / voice / background.

Re-run the preprocess cell after downloading new AudioSet wavs. Set `FORCE_REPROCESS = True` to rebuild everything.

In [1]:
%matplotlib inline

from pathlib import Path
import sys

import ipywidgets as widgets
import librosa
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, clear_output, display
from matplotlib.image import imread

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "source_separation").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from source_separation.audioset_preprocess import (
    ClipRecord,
    clips_for_label,
    labels_with_clips,
    load_manifest,
    paths_for_clip,
    preprocess_all,
)

AUDIO_ROOT = REPO_ROOT / "datasets/audioset/audio"
PROCESSED_ROOT = REPO_ROOT / "datasets/audioset/processed"
CKPT_PATH = REPO_ROOT / "checkpoints/unet_run1/unet_voice_sep.pt"
GRID_COLS = 3

In [2]:
FORCE_REPROCESS = False

if not CKPT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")

summary = preprocess_all(
    AUDIO_ROOT,
    PROCESSED_ROOT,
    CKPT_PATH,
    force=FORCE_REPROCESS,
)
print("Done:", summary)
print(f"Manifest: {PROCESSED_ROOT / 'manifest.csv'}")

preprocess: 100%|██████████| 156/156 [00:03<00:00, 46.03it/s]

FAILED Crowd/Db06mHi_gaY_10.0s.wav: Argument #4: Padding size should be less than the corresponding input dimension, but got: padding (512, 512) at dimension 2 of input [1, 1, 125]
FAILED Gargling/3EOMe7CsL30_10.0s.wav: Argument #4: Padding size should be less than the corresponding input dimension, but got: padding (512, 512) at dimension 2 of input [1, 1, 125]
FAILED Heart sounds, heartbeat/kLdHpgN9kwU_10.0s.wav: Argument #4: Padding size should be less than the corresponding input dimension, but got: padding (512, 512) at dimension 2 of input [1, 1, 125]
FAILED Sniff/0qUXJz4HH_E_10.0s.wav: Argument #4: Padding size should be less than the corresponding input dimension, but got: padding (512, 512) at dimension 2 of input [1, 1, 125]
FAILED Whoop/QVO6U85x9hw_10.0s.wav: Argument #4: Padding size should be less than the corresponding input dimension, but got: padding (512, 512) at dimension 2 of input [1, 1, 125]
Done: {'processed': 0, 'skipped': 151, 'failed': 5}
Manifest: /home/luusam

In [5]:
def _tile_image(path, ax, title=""):
    img = imread(path)
    # PNGs from matplotlib: row 0 = top of figure = high Hz; match with origin="upper"
    ax.imshow(img, aspect="auto", origin="upper")
    ax.set_title(title, fontsize=7)
    ax.set_xticks([])
    ax.set_yticks([])


def draw_comparison_grid(clips: list[ClipRecord], cols: int = GRID_COLS) -> None:
    n = len(clips)
    if n == 0:
        print("No clips for this label.")
        return
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows * 2, cols, figsize=(4 * cols, 2.8 * rows), squeeze=False)

    for i, rec in enumerate(clips):
        col = i % cols
        row_pair = (i // cols) * 2
        p = paths_for_clip(rec)
        _tile_image(p["voice_spec"], axes[row_pair, col], "voice")
        _tile_image(p["bg_spec"], axes[row_pair + 1, col], "bg")
        axes[row_pair + 1, col].set_xlabel(
            f"{rec.stem}\nr={rec.voice_energy_ratio:.2f}",
            fontsize=6,
        )

    for j in range(n, rows * cols):
        col = j % cols
        row_pair = (j // cols) * 2
        axes[row_pair, col].axis("off")
        axes[row_pair + 1, col].axis("off")

    fig.suptitle("Voice (top) vs background (bottom) — look for spectral overlap", fontsize=10)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def draw_detail(rec: ClipRecord) -> None:
    p = paths_for_clip(rec)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, key, title in zip(axes, ["mix_spec", "voice_spec", "bg_spec"], ["mix", "voice", "bg"]):
        _tile_image(p[key], ax, title)
    fig.suptitle(f"{rec.sub_label} / {rec.stem}", fontsize=11)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

    y_mix, sr = librosa.load(str(p["mix"]), sr=None, mono=True)
    y_voice, _ = librosa.load(str(p["voice"]), sr=None, mono=True)
    y_bg, _ = librosa.load(str(p["bg"]), sr=None, mono=True)
    print(f"duration={rec.duration_sec:.2f}s  voice_energy_ratio={rec.voice_energy_ratio:.3f}")
    display(Audio(y_mix, rate=sr))
    display(Audio(y_voice, rate=sr))
    display(Audio(y_bg, rate=sr))

In [ ]:
label_options = labels_with_clips(PROCESSED_ROOT)
if not label_options:
    raise RuntimeError(
        "No processed clips found. Run the preprocess cell above first."
    )

label_dropdown = widgets.Dropdown(
    options=[(f"{name} ({count})", name) for name, count in label_options],
    description="Sub-label:",
    layout=widgets.Layout(width="420px"),
)
clip_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    description="Clip:",
    continuous_update=False,
)
grid_out = widgets.Output()
detail_out = widgets.Output()

_state = {"clips": []}


def refresh_ui(*_):
    sub_label = label_dropdown.value
    clips = clips_for_label(PROCESSED_ROOT, sub_label)
    _state["clips"] = clips
    clip_slider.max = max(0, len(clips) - 1)
    if clip_slider.value > clip_slider.max:
        clip_slider.value = 0

    with grid_out:
        clear_output(wait=True)
        draw_comparison_grid(clips)

    with detail_out:
        clear_output(wait=True)
        if clips:
            draw_detail(clips[clip_slider.value])


label_dropdown.observe(refresh_ui, names="value")
clip_slider.observe(refresh_ui, names="value")

refresh_ui()
display(
    widgets.VBox(
        [
            label_dropdown,
            clip_slider,
            widgets.HTML("<b>Comparison grid</b> (cached specs)"),
            grid_out,
            widgets.HTML("<b>Selected clip</b>"),
            detail_out,
        ]
    )
)